# What Is a Signal?

This notebook starts the course at the physical and mathematical beginning: a signal is just a quantity that changes over time. We use sine waves to build intuition for amplitude, frequency, and phase because they are the atoms of DSP.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from rf_utils import *
from IPython.display import Audio, Markdown, display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

%matplotlib widget

plt.rcParams.update({
    "figure.figsize": (12, 4),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})


## One Tone, Three Knobs

For a cosine,

$$s(t) = A \cos(2 \pi f t + \phi)$$

amplitude changes height, frequency changes how fast it oscillates, and phase shifts where the cycle starts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
audio_out = audio_output_widget()

def update_tone(freq=440.0, amplitude=0.8, phase=0.0):
    t, sig = generate_tone(freq=freq, duration=0.02, fs=44_100, amplitude=amplitude, phase=phase)
    axes[0].clear()
    axes[1].clear()
    plot_waveform(sig, fs=44_100, ax=axes[0], title="Waveform")
    plot_spectrum(sig, fs=44_100, ax=axes[1], title="Spectrum")
    axes[1].set_xlim(0, 2000)
    axes[1].set_ylim(-100, 5)
    fig.canvas.draw_idle()
    refresh_audio_widget(audio_out, sig, rate=44_100)

controls = widgets.interactive(
    update_tone,
    freq=float_slider(min_value=20, max_value=4000, step=10, value=440, description="Freq Hz"),
    amplitude=float_slider(min_value=0, max_value=1, step=0.05, value=0.8, description="Amp"),
    phase=float_slider(min_value=0, max_value=2 * np.pi, step=0.1, value=0.0, description="Phase"),
)
display(controls, audio_out)


## Phase Only Matters Relative to Something

A single tone shifted in phase sounds the same to your ear, but phase becomes important when two signals interact. Overlaying two equal tones makes the shift visible immediately.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

def update_phase_difference(phase=0.0):
    ax.clear()
    t, sig1 = generate_tone(freq=300, duration=0.02, fs=44_100, amplitude=1.0, phase=0.0)
    _, sig2 = generate_tone(freq=300, duration=0.02, fs=44_100, amplitude=1.0, phase=phase)
    ax.plot(t * 1e3, sig1, label="Reference")
    ax.plot(t * 1e3, sig2, label="Shifted", alpha=0.8)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude")
    ax.set_title(f"Phase offset = {phase:.2f} rad")
    ax.legend()
    fig.canvas.draw_idle()

controls = widgets.interactive(
    update_phase_difference,
    phase=float_slider(min_value=0, max_value=2 * np.pi, step=0.1, value=0.0, description="Phase"),
)
display(controls)


## What to Try

- Double the frequency and notice that the pitch rises while the waveform compresses in time.
- Set amplitude near zero and confirm that the spectrum peak drops with it.
- Sweep phase through `pi` while comparing two overlaid tones and watch the shift without hearing a pitch change.

## Key Takeaway

A signal is just a time-varying quantity, but sinusoids are the special case that make every later DSP concept manageable. If you can reason about amplitude, frequency, and phase, the rest of the course has a base to stand on.